# POC for distributed data alignment


## Metrics for evaluating synthetic data

We use Adult as typical benchmark dataset.


In [1]:
import os
import sys
from pathlib import Path

base_path = Path(os.path.abspath("")).parent
sys.path.append(base_path.as_posix())

In [3]:
from privfusion.metrics import metrics_fidelity, metrics_privacy
from privfusion.tabular_data import utils

In [4]:
FIDELITY_METRICS = ["KSComplement", "TVComplement", "WD", "JSD"]
PRIVACY_METRICS = ["DCR", "NNDR"]

In [5]:
NUMERICAL_COLUMNS = ["age", "educational-num", "capital-gain", "capital-loss", "hours-per-week"]
CATEGORICAL_COLUMNS = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "gender",
    "native-country",
    "income",
]

In [6]:
def compute_fidelity_metrics(df_real, df_synth, list_metrics):
    results = {}
    for metric in list_metrics:
        # numerical columns or probability distribution of the columns
        if metric == "WD":
            wd = metrics_fidelity.compute_WD(df_real[NUMERICAL_COLUMNS], df_synth[NUMERICAL_COLUMNS])
            results[metric] = wd
        # numerical columns or probability distribution of the columns
        elif metric == "JSD":
            jsd = metrics_fidelity.compute_JSD(df_real[NUMERICAL_COLUMNS], df_synth[NUMERICAL_COLUMNS])
            results[metric] = jsd
        # numerical
        elif metric == "KSComplement":
            ks = metrics_fidelity.compute_KSComplement(df_real[NUMERICAL_COLUMNS], df_synth[NUMERICAL_COLUMNS])
            results[metric] = ks
        # categorical
        elif metric == "TVComplement":
            tv = metrics_fidelity.compute_TVComplement(df_real[CATEGORICAL_COLUMNS], df_synth[CATEGORICAL_COLUMNS])
            results[metric] = tv
        else:
            raise ValueError(f"{metric} is not a valid metric")
    return results

In [7]:
def compute_privacy_metrics(
    df_real,
    df_synth,
    list_metrics,
    data_percent: float = 0.1,
):
    results = {}
    for metric in list_metrics:
        # numerical
        if metric == "DCR":
            dcr = metrics_privacy.compute_DCR(df_real[NUMERICAL_COLUMNS], df_synth[NUMERICAL_COLUMNS], data_percent)
            results[metric] = dcr
        # numerical
        elif metric == "NNDR":
            nndr = metrics_privacy.compute_NNDR(df_real[NUMERICAL_COLUMNS], df_synth[NUMERICAL_COLUMNS], data_percent)
            results[metric] = nndr
    return results

In [ ]:
real_data_path = base_path.joinpath("data/adult_dataset/adult.csv").as_posix()
synth_data_path = base_path.joinpath("output/adult_dataset/adult_gen_samples.csv").as_posix()


df_real = utils.read_data(real_data_path)
df_synth = utils.read_data(synth_data_path)

## Fidelity Metrics

In [ ]:
fidelity_metrics = compute_fidelity_metrics(df_real=df_real, df_synth=df_synth, list_metrics=FIDELITY_METRICS)

In [ ]:
fidelity_metrics

{'KSComplement': 0.806386474616873,
 'TVComplement': 0.6469540930081352,
 'WD': 235.39066521298514,
 'JSD': 0.11788261400289755}

## Privacy Metrics

In [ ]:
privacy_metrics = compute_privacy_metrics(df_real=df_real, df_synth=df_synth, list_metrics=PRIVACY_METRICS)

In [ ]:
privacy_metrics

{'DCR': 0.8120470776163555, 'NNDR': 0.4806261830761544}